# Keras → ONNX → EZKL pipeline

Notebook version of `model.py`: build a small MNIST-style CNN, export to ONNX, then (optionally) run the ezkl proof pipeline.

Run from the `reference-pipeline` directory with the project venv active:

```bash
source ../.venv/bin/activate
jupyter notebook model.ipynb
```

In [ ]:
# import json
# import ezkl
import numpy as np
import tensorflow as tf
import tf2onnx

## Build the model

Use the **Functional API** with an explicit `Reshape((21632,))` after Conv2D — required for ezkl (see repo README, Problems 2–4).

The Sequential + `Flatten()` variant is kept below as a comment for reference; it fails tf2onnx or ezkl depending on export settings.

In [ ]:
# Sequential API model (commented — fails tf2onnx / ezkl in this pipeline)
#model = tf.keras.Sequential([
#     tf.keras.layers.Input(shape=(28, 28, 1)),
#     tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
#     tf.keras.layers.Flatten(),
#     tf.keras.layers.Dense(10, activation="softmax"),
# ])

# Functional API model
inputs = tf.keras.Input(shape=(28, 28, 1), batch_size=1)
print(inputs)
x = tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu")(inputs)
print(x)
x = tf.keras.layers.Reshape((21632,))(x)
print(x)
outputs = tf.keras.layers.Dense(10, activation="softmax")(x)
print(outputs)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

print(model.outputs)
print(model.output_shape)
# model.summary()


## Export Keras → ONNX

For a static batch in ONNX (needed by ezkl), pass `input_signature` to `from_keras` — see README Problem 4.

In [ ]:
# input_signature = [tf.TensorSpec([1, 28, 28, 1], tf.float32, name="input")]
# onnx_model, _ = tf2onnx.convert.from_keras(model, input_signature=input_signature)

onnx_model, _ = tf2onnx.convert.from_keras(model)

with open("network.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

print("Wrote network.onnx (", len(onnx_model.SerializeToString()), "bytes)")

## EZKL pipeline (optional)

Uncomment the imports at the top and the cells below to run the full proof pipeline. `prove` takes minutes on CPU for this model (~913k circuit rows).

In [ ]:
# ezkl.gen_settings("network.onnx")
# ezkl.calibrate_settings("network.onnx", "settings.json", target="resources")
# ezkl.compile_circuit("network.onnx", "network.ezkl", "settings.json")

# if not __import__("os").path.exists("kzg.srs"):
#     ezkl.get_srs("settings.json", srs_path="kzg.srs")

# ezkl.setup("network.ezkl", "vk.key", "pk.key", "kzg.srs")

In [ ]:
# flat layout required by ezkl (nested arrays fail to deserialize)
# sample_input = np.random.default_rng(0).random((1, 28, 28, 1), dtype=np.float32)
# sample_output = model(sample_input).numpy()
# with open("input.json", "w") as f:
#     json.dump(
#         {
#             "input_data": [sample_input.reshape(-1).tolist()],
#             "output_data": [sample_output.reshape(-1).tolist()],
#         },
#         f,
#     )

# ezkl.gen_witness(
#     data="input.json",
#     model="network.ezkl",
#     output="witness.json",
# )

# ezkl.prove(
#     witness="witness.json",
#     model="network.ezkl",
#     pk_path="pk.key",
#     proof_path="proof.json",
#     srs_path="kzg.srs",
# )

# ezkl.verify(
#     proof_path="proof.json",
#     settings_path="settings.json",
#     vk_path="vk.key",
#     srs_path="kzg.srs",
# )